# 第 8 周顶点 — 自主多代理交易查找器（Hannaford 与 Market Basket）

## 练习目标（理念）

用第 8 周的**多智能体编排**模式，对比两家真实本地超市（Hannaford / Market Basket）的购物篮价格，并让 **Reporter** 写出购物建议。

## 和第 8 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Planning Agent（规划代理） | `PlanningAgent` 调度扫描 → 两家定价 → 比较 |
| 专业子代理 | Scanner（购物篮）、两名 Pricer、Reporter |
| LLM 估价 vs 实时数据 | 价格由 `gpt-4o-mini` **估计**，非店铺 API |

## 怎么跑

1. 确保仓库里有 `community-contributions/NicholasDean/week8/grocery_agents.py`，以及 `week6/pricer/items.py`（用于定位仓库根）
2. 配置 OpenAI 密钥（`grocery_agents.py` 内 `load_dotenv` + `OpenAI()`）
3. 从上到下运行：先出比价表，再看总计与文字推荐

> 价格为 `gpt-4o-mini` **估计**，而非实时数据 — 生产版本可把实时 API/传单接入 Scanner 与 Pricer；**代理架构不变**。


In [1]:
# ========== 规划循环：定位仓库 → 导入代理 → 跑比价 → 用表格展示 ==========

# 标准库 sys：把自定义模块路径插进 Python 模块搜索路径
import sys
# pathlib.Path：跨平台路径，用来向上查找仓库根目录
from pathlib import Path
# pandas：把 plan["rows"] 变成可读的 DataFrame 表格
import pandas as pd

# 从当前目录及其父目录里找含 week6/pricer/items.py 的路径，当作仓库根 REPO
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week6/pricer/items.py").exists())
# 把作者 week8 目录加入 sys.path，才能 import grocery_agents
sys.path.insert(0, str(REPO / "community-contributions/NicholasDean/week8"))
# 导入规划代理与记者代理（noqa：导入在 path 修改之后，lint 可忽略顺序警告）
from grocery_agents import PlanningAgent, ReportAgent  # noqa: E402

# 跑完整流水线：Scanner 购物篮 → 两家 Pricer → 逐项比较（结果进 plan）
plan = PlanningAgent().run()           # Scanner -> two Pricers -> compare
# 把每行比价结果做成 DataFrame，在笔记本里直接展示
pd.DataFrame(plan["rows"])


,item,Hannaford,Market Basket,cheaper,savings
0,1 gallon whole milk,4.49,3.49,Market Basket,1.0
1,one dozen large eggs,2.99,2.99,Hannaford,0.0
2,loaf of white sandwich bread,2.49,1.99,Market Basket,0.5
3,2 lb bananas,1.99,1.49,Market Basket,0.5
4,1 lb boneless chicken breast,4.99,5.99,Hannaford,1.0
5,8 oz block cheddar cheese,3.49,2.49,Market Basket,1.0
6,12 oz bag ground coffee,7.99,6.99,Market Basket,1.0
7,1 lb butter,3.89,4.29,Hannaford,0.4
8,64 oz orange juice,3.49,3.99,Hannaford,0.5
9,18 oz peanut butter,3.29,3.49,Hannaford,0.2


## 总计和推荐（记者代理）

上一格得到逐项比价表；本格打印两家购物篮总价、按「每件选更便宜店」算出的最优拆分，并调用 **ReportAgent** 生成自然语言购物建议。


In [2]:
# ========== 输出总计 + 调用 Reporter 写购物建议 ==========

# 打印两家店购物篮合计（字典：店名 → 总价）
print("Basket totals:", plan["totals"])
# 打印最优拆分总价：每件商品都在更便宜的那家买
print("Optimal (buy each item at its cheaper store): $", plan["optimal"])
# 空行，方便和下面的长文本推荐分开看
print()
# 把完整 plan 交给 Reporter，让 LLM 写 4–5 句购物建议（英文 prompt 在 grocery_agents 内）
print(ReportAgent().run(plan))


Basket totals: {'Hannaford': 39.1, 'Market Basket': 37.2}
Optimal (buy each item at its cheaper store): $ 35.1



For an overall cheaper grocery shopping experience, Market Basket is the better option, offering total savings of $1.90 compared to Hannaford. Notable price differences include 1 gallon of whole milk, which is $1.00 less at Market Basket, and a loaf of white sandwich bread that saves you $0.50 at Market Basket as well. Additionally, 2 lb of bananas also has a $0.50 savings at Market Basket. To maximize savings, consider shopping at Market Basket.
